# Tick 按日打包文件 — 读测试

验证 `tick_by_day/YYYY/MM/DD.parquet`（单文件含当日全部股票）的读取：
1. 文件基本信息（大小 / row-group / 行数）
2. 读取全天数据
3. 读取**指定股票**（谓词下推，不解压整文件）
4. 获取文件的**股票列表**（只读 `stock_code` 列，开销小）
5. row-group 区间（证明单股读取可跳过无关 row-group）

In [ ]:
import sys
sys.path.insert(0, r"D:\proj\data_collect")  # 项目根目录，按需调整

import pandas as pd
import pyarrow.parquet as pq
from data_collect.config import get_tick_config
from data_collect.jobs.a_share_tick import read_tick, _day_file_path

trade_date = "20260624"
print("tick base_dir:", get_tick_config()["base_dir"])

## 1. 文件基本信息

In [ ]:
f = _day_file_path(trade_date)
print("文件:", f)
print("存在:", f.exists())
assert f.exists() and f.stat().st_size > 0, "当日打包文件不存在，请先采集该交易日"

pf = pq.ParquetFile(str(f))
print(f"大小: {f.stat().st_size/1e6:.2f} MB | row_groups: {pf.num_row_groups} | 总行数: {pf.metadata.num_rows:,}")

## 2. 读取全天数据

冷数据较大，整天读取会把全部股票载入内存，按需使用。

In [ ]:
df = read_tick(trade_date)
print("shape:", df.shape, "| 股票数:", df["stock_code"].nunique())
df.head()

## 3. 读取指定股票

`read_tick(date, stock_code)` 走 parquet 谓词下推：只返回该股的行，且因 row-group 按代码段有序，
pyarrow 会跳过不含该股的 row-group，不会解压整文件。

In [ ]:
one = read_tick(trade_date, "000001.SZ")
print("shape:", one.shape, "| 唯一股票:", list(one["stock_code"].unique()))
one.head()

## 4. 获取当日文件的股票列表

只读 `stock_code` 一列（列式存储，开销远小于读全表），去重排序。

In [ ]:
codes_tbl = pq.read_table(str(f), columns=["stock_code"])
stock_list = sorted(set(codes_tbl.column("stock_code").to_pylist()))
print("股票数:", len(stock_list))
print("前 10:", stock_list[:10])
print("后 5 :", stock_list[-5:])

## 5. row-group 区间（单股读取为何高效）

每个 row-group 的 `stock_code` min/max **有序且不重叠** → 单股 filter 只命中一个 row-group。

In [ ]:
for i in range(pf.num_row_groups):
    st = pf.metadata.row_group(i).column(0).statistics  # col0 = stock_code
    print(f"rg{i}: [{st.min} .. {st.max}]  rows={pf.metadata.row_group(i).num_rows:,}")